# 🧪 Case File 10: Write the Name Yourself

Welcome to the custom-expression basement. This lab shows what must be owned when a normal UDF is not expressive enough: semantics, null behavior, interpreted evaluation, generated evaluation, registration, and physical-plan integration.

**Mission Objective:** test one intentionally boring deterministic operation through three separate layers: a reference/interpreted implementation, an equivalent native generated expression, and WholeStageCodegen plan integration.

**Maintenance Guardrail:** a real custom Catalyst expression is Scala/JVM code registered through Spark extensions. This PySpark notebook documents and tests the contract, but it does not pretend that a Python function has become a Catalyst expression.


### Step 1: Define the Spark test session
The session lets us test the native generated equivalent and inspect WholeStageCodegen integration separately from expression correctness.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

spark = (SparkSession.builder
    .master("local[2]")
    .appName("case-file-10-write-the-name-yourself")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:39:45 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:39:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 06:39:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Write the expression contract
The following Scala skeleton is the shape of a real unary Catalyst expression. It declares a type contract, null-safe interpreted evaluation, generated evaluation, and a function name. It is shown as source evidence; compiling and registering it requires a separate Scala/JAR build.


In [2]:
scala_expression_skeleton = r'''
case class AddOne(child: Expression)
    extends UnaryExpression
    with ExpectsInputTypes {

  override def inputTypes = Seq(LongType)
  override def dataType: DataType = LongType
  override def nullable: Boolean = child.nullable

  override protected def nullSafeEval(input: Any): Any =
    input.asInstanceOf[Long] + 1L

  override protected def doGenCode(
      ctx: CodegenContext, ev: ExprCode): ExprCode = {
    defineCodeGen(ctx, ev, value => s"$value + 1L")
  }

  override def prettyName: String = "add_one"
}

// SparkSessionExtensions registration shape:
val addOneFunction = (
  FunctionIdentifier("add_one"),
  new ExpressionInfo(classOf[AddOne].getName, "add_one"),
  (children: Seq[Expression]) => AddOne(children.head)
)

extensions.injectFunction(addOneFunction)
'''
print(scala_expression_skeleton)



case class AddOne(child: Expression)
    extends UnaryExpression
    with ExpectsInputTypes {

  override def inputTypes = Seq(LongType)
  override def dataType: DataType = LongType
  override def nullable: Boolean = child.nullable

  override protected def nullSafeEval(input: Any): Any =
    input.asInstanceOf[Long] + 1L

  override protected def doGenCode(
      ctx: CodegenContext, ev: ExprCode): ExprCode = {
    defineCodeGen(ctx, ev, value => s"$value + 1L")
  }

  override def prettyName: String = "add_one"
}

// SparkSessionExtensions registration shape:
val addOneFunction = (
  FunctionIdentifier("add_one"),
  new ExpressionInfo(classOf[AddOne].getName, "add_one"),
  (children: Seq[Expression]) => AddOne(children.head)
)

extensions.injectFunction(addOneFunction)



### Step 3: Test the interpreted semantics
The reference implementation is deliberately boring: add one to a nullable long. We test ordinary values, `NULL`, negative values, and the long boundary that exposes overflow behavior.


In [3]:
def interpreted_add_one(value):
    if value is None:
        return None
    return ((value + 1 + 2**63) % 2**64) - 2**63

test_values = [None, -2, -1, 0, 1, 2, 2**63 - 1]

interpreted_results = [
    (value, interpreted_add_one(value))
    for value in test_values
]

for value, result in interpreted_results:
    print(f"interpreted input={value!r}, output={result!r}")


interpreted input=None, output=None
interpreted input=-2, output=-1
interpreted input=-1, output=0
interpreted input=0, output=1
interpreted input=1, output=2
interpreted input=2, output=3
interpreted input=9223372036854775807, output=-9223372036854775808


### Step 4: Test the generated-expression equivalent
PySpark cannot register the Scala class above by itself, so we express the same semantics with native Spark SQL. This exercises the generated-expression path available in the current runtime, while keeping the custom Catalyst integration claim separate.


In [4]:
rows = [(value,) for value in test_values]

inputs = (
    spark.createDataFrame(rows, ["value"])
    .withColumn("value", F.col("value").cast(LongType()))
)

generated_equivalent = inputs.select(
    "value",
    (F.col("value") + F.lit(1)).alias("result"),
)

generated_results = [
    (row["value"], row["result"])
    for row in generated_equivalent.collect()
]

print("generated-equivalent results:", generated_results)
print("interpreted/native results agree:", interpreted_results == generated_results)

assert interpreted_results == generated_results


generated-equivalent results: [(None, None), (-2, -1), (-1, 0), (0, 1), (1, 2), (2, 3), (9223372036854775807, -9223372036854775808)]
interpreted/native results agree: True


### Step 5: Test the three layers separately
The native expression can participate in WholeStageCodegen. That confirms the third layer for the equivalent expression, not for the uncompiled Scala class. The codegen setting is not a universal switch for every expression implementation.


In [5]:
print("=== Native equivalent physical plan: codegen on ===")
spark.conf.set("spark.sql.codegen.wholeStage", "true")
generated_equivalent_on = inputs.select("value", (F.col("value") + F.lit(1)).alias("result"))
generated_equivalent_on.explain("formatted")
print("=== Native equivalent generated code ===")
generated_equivalent_on.explain("codegen")
import random
random_values = random.Random(42).choices(range(-1000, 1000), k=25)
random_inputs = spark.createDataFrame([(value,) for value in random_values], ["value"]).withColumn("value", F.col("value").cast(LongType()))
random_results = [(row["value"], row["result"]) for row in random_inputs.select("value", (F.col("value") + 1).alias("result")).collect()]
print("randomized semantic cases agree:", all(interpreted_add_one(value) == result for value, result in random_results))
print("=== WholeStageCodegen integration: codegen off ===")
spark.conf.set("spark.sql.codegen.wholeStage", "false")
generated_equivalent_off = inputs.select("value", (F.col("value") + F.lit(1)).alias("result"))
generated_equivalent_off.explain("formatted")
spark.conf.set("spark.sql.codegen.wholeStage", "true")
print("Layer 1 — interpreted contract: tested")
print("Layer 2 — generated-equivalent semantics: tested")
print("Layer 3 — native WholeStageCodegen integration on/off: inspected on freshly built queries")
print("Custom Catalyst class integration: requires compiled Scala/JAR extension")


=== Native equivalent physical plan: codegen on ===
== Physical Plan ==
* Project (2)
+- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [1]: [value#0L]
Arguments: [value#0L], MapPartitionsRDD[4] at applySchemaToPythonRDD at DirectMethodHandleAccessor.java:103, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [2]: [value#0L, (value#0L + 1) AS result#3L]
Input [1]: [value#0L]


=== Native equivalent generated code ===


Found 1 WholeStageCodegen subtrees.
== Subtree 1 / 1 (maxMethodCodeSize:110; maxConstantPoolSize:119(0.18% used); numInnerClasses:0) ==
*(1) Project [value#0L, (value#0L + 1) AS result#3L]
+- *(1) Scan ExistingRDD[value#0L]

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private scala.collection.Iterator rdd_input_0;
/* 010 */   private org.apache.spark.sql.catalyst.expressions.codegen.UnsafeRowWriter[] rdd_mutableStateArray_0 = new org.apache.spark.sql.catalyst.expressions.codegen.UnsafeRowWriter[3];
/* 011 */
/* 012 */   public GeneratedIteratorForCodegenStage1(Object[] references) {
/* 013 */     this.references = re

randomized semantic cases agree: True
=== WholeStageCodegen integration: codegen off ===
== Physical Plan ==
Project (2)
+- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [1]: [value#0L]
Arguments: [value#0L], MapPartitionsRDD[4] at applySchemaToPythonRDD at DirectMethodHandleAccessor.java:103, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [value#0L, (value#0L + 1) AS result#7L]
Input [1]: [value#0L]


Layer 1 — interpreted contract: tested
Layer 2 — generated-equivalent semantics: tested
Layer 3 — native WholeStageCodegen integration on/off: inspected on freshly built queries
Custom Catalyst class integration: requires compiled Scala/JAR extension


### Step 6: Record the maintenance bill
A custom expression must be compiled, registered, placed on the Spark classpath, and tested across Spark versions. Spark 4.2 marks `SparkSessionExtensions` experimental/unstable, so source and binary compatibility are part of the ownership cost.


In [6]:
maintenance_items = [
    "Scala/JVM build and JAR packaging",
    "SparkSessionExtensions registration",
    "classpath and deployment management",
    "interpreted versus generated semantic tests",
    "Spark-version compatibility matrix",
]
print("Maintenance items:")
for item in maintenance_items:
    print("-", item)


Maintenance items:
- Scala/JVM build and JAR packaging
- SparkSessionExtensions registration
- classpath and deployment management
- interpreted versus generated semantic tests
- Spark-version compatibility matrix


# 📊 Post-Lab Analysis: Write the Name Yourself

This case file keeps three claims separate. The reference function establishes intended semantics. The native Spark expression demonstrates a generated evaluation path with equivalent behavior. A real custom Catalyst expression still requires a compiled Scala/JVM extension before its WholeStageCodegen integration can be judged.

### 1. A UDF Gives Spark a Function; an Expression Gives Spark Semantics

A custom expression declares children, types, null behavior, interpreted evaluation, and generated evaluation. That is a deeper contract than asking Spark to call an opaque function.

### 2. Two Paths Must Tell the Same Story

The interpreted and generated paths must agree for ordinary values, `NULL`, edge values, overflow behavior, and randomized inputs. This notebook checks nullable values and the signed 64-bit overflow boundary explicitly.

### 3. Integration Is a Separate Verdict

The native equivalent joins WholeStageCodegen because Spark already understands it. The custom Scala skeleton cannot receive that verdict until it is compiled, registered, and placed on the exact runtime classpath.

Custom Catalyst work is for hot, stable paths worth owning. It is not the automatic next step for every slow UDF.
